In [ ]:
# Imports and setup
import json
import os
import sys
from typing import List, Optional, Dict
import gc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import auc, roc_curve
from tqdm import tqdm

# Import memory optimization helpers
sys.path.append('..')
try:
    from memory_optimized_notebook import load_dataframe, clear_plt, print_memory_usage
except ImportError:
    # Fallback if module not available
    def print_memory_usage(): pass
    def clear_plt(): 
        plt.close('all')
        gc.collect()
    load_dataframe = pd.read_csv

# Force garbage collection at start
gc.collect()

sys.path.append('..')
sys.path.append('../..')
from plotting_notebooks.plotting_utils import (
    load_in_combined_results,
    load_monitoring_data,
    plot_markov_analysis,
    plot_roc_curves,
    plot_suspiciousness_distributions,
    plot_cost_vs_safety_curves,
    get_roc_curve,
    interpolate_tpr,
    prepare_monitoring_data,
    merge_monitoring_with_combined_results,
    filter_and_prepare_data,
    is_in_training_data,
)
from simulate_markov_utils import plot_markov_df
from utils.analysis_utils import extract_verdict
from utils.path_utils import resolve_path

# Add memory tracking
print_memory_usage()


In [ ]:
# Constants and configuration
MAIN_AGENT_MODELS = ['claude-3-5-sonnet-20241022']
MONITOR_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022', 'o3-mini', 'o1', 'claude-3-5-haiku-20241022']
DEFAULT_TRANSCRIPT_TYPES = ["transcripts", "transcripts_no_hint", "transcripts_no_secondary_task"]
DOWNSAMPLE_FACTORS = [1, 2, 3, 5, 10, 15]
BACK_FRACS = [0.1, 0.2, 0.5, 0.8]
NUM_MONITORS = 10

# Step mapping for fine-tuned models
FT_MONITOR_STEP_70 = 'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFx2l:ckpt-step-70'
FT_STEP_MAPPING = {
    "gpt-4o-2024-08-06": 0,
    'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFx2l:ckpt-step-70': 70,
    'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFtC0:ckpt-step-140': 140,
    'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFa40': 210,
    'ft:gpt-4o-2024-08-06:scale-safety-research-1:unbalanced-data:BDXppMow:ckpt-step-56': 56,
    'ft:gpt-4o-2024-08-06:scale-safety-research-1:unbalanced-data:BDXpp5YL:ckpt-step-112': 112,
    'ft:gpt-4o-2024-08-06:scale-safety-research-1:unbalanced-data:BDXppDFa': 168,
}
FT_MONITOR_MODELS = list(FT_STEP_MAPPING.keys())

# Cache for combined_results (modified to be more memory-efficient)
_combined_results_cache = {}


In [ ]:
# Helper functions for data loading

def load_combined_results(include_failed_attempts: bool, include_failed_main_task: bool) -> pd.DataFrame:
    """Load and cache combined_results."""
    cache_key = f"{include_failed_attempts}_{include_failed_main_task}"
    if cache_key in _combined_results_cache:
        print(f"Using cached combined_results for {cache_key}")
        return _combined_results_cache[cache_key]
    combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)
    _combined_results_cache[cache_key] = combined_results
    
    # Add memory tracking
    print_memory_usage()
    return combined_results

def get_downsampled_responses(x, downsample_factor, back_frac=1.0):
    """Extract responses with specified downsampling factor."""
    all_indices = list(x.values())[::downsample_factor] + [list(x.values())[-1]]
    back_indices = all_indices[-int(len(all_indices) * back_frac):]
    return back_indices

def prepare_turn_by_turn_data(base_dir: str, 
                             downsample_factors: List[int] = DOWNSAMPLE_FACTORS, 
                             thresholds_to_consider = [4, 9, 14, 19, 29, 39], 
                             nth_highest_indices: List[int] = [2, 3, 5], 
                             suffix: str = 'turn_by_turn') -> pd.DataFrame:
    """Load and prepare the turn-by-turn monitoring data."""
    # Load monitoring data
    turn_by_turn_df = load_monitoring_data(base_dir=base_dir)
    
    # Calculate downsampled max suspiciousness scores
    for downsample_factor in downsample_factors:
        turn_by_turn_df[f'max_suspiciousness_{downsample_factor}_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(max([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 
                               for y in get_downsampled_responses(x, downsample_factor)]))
        )
        turn_by_turn_df[f'max_suspiciousness_usage_{downsample_factor}_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, downsample_factor)]))
        )
    
    # Force garbage collection after intensive processing
    gc.collect()
    
    for back_frac in BACK_FRACS:
        turn_by_turn_df[f'max_suspiciousness_back_{back_frac}_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(max([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 
                               for y in get_downsampled_responses(x, 1, back_frac)]))
        )
        turn_by_turn_df[f'mean_suspiciousness_back_{back_frac}_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(np.mean([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 
                               for y in get_downsampled_responses(x, 1, back_frac)]))
        )
        turn_by_turn_df[f'max_suspiciousness_back_{back_frac}_usage_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, 1, back_frac)]))
        )
        turn_by_turn_df[f'mean_suspiciousness_back_{back_frac}_usage_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, 1, back_frac)]))
        )
    gc.collect()
        
    # Calculate fraction above thresholds
    for threshold in thresholds_to_consider:
        turn_by_turn_df[f'frac_above_{threshold}_{suffix}'] = 100 * turn_by_turn_df.data.apply(
            lambda x: float(np.mean([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') > threshold or 0.0 
                                   for y in get_downsampled_responses(x, 1)]))
        )

        turn_by_turn_df[f'frac_above_{threshold}_usage_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, 1)]))
        )
    
    # Force garbage collection after intensive processing
    gc.collect()
    
    # Calculate nth highest suspiciousness scores
    for n in nth_highest_indices:
        turn_by_turn_df[f'{n}th_highest_suspiciousness_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: sorted([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 
                            for y in list(x.values())], reverse=True)[n-1] if len(x) >= n else 0.0
        )

        turn_by_turn_df[f'{n}th_highest_suspiciousness_usage_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, 1)]))
        )
    
    # Calculate final suspiciousness and usage
    turn_by_turn_df[f'final_suspiciousness_{suffix}'] = turn_by_turn_df.data.apply(
        lambda x: float(extract_verdict(x[str(max([int(k) for k in x.keys()]))]["content"] 
                                      if isinstance(x[str(max([int(k) for k in x.keys()]))], dict) 
                                      and "content" in x[str(max([int(k) for k in x.keys()]))] 
                                      else x[str(max([int(k) for k in x.keys()]))], tag_word='verdict'))
    )
    
    turn_by_turn_df[f'final_usage_{suffix}'] = turn_by_turn_df.data.apply(
        lambda x: float(x[str(max([int(k) for k in x.keys()]))]["usage"][0]["cost_usd"])
    )
    
    # Add side_task_evaluation column
    turn_by_turn_df['side_task_evaluation'] = turn_by_turn_df.transcript_type != 'transcripts_no_secondary_task'
    
    # Memory tracking and cleanup
    print_memory_usage()
    gc.collect()
    
    return turn_by_turn_df

In [ ]:
turn_by_turn_df_1 = prepare_turn_by_turn_data(base_dir='monitor_responses/turn_by_turn_monitoring')
turn_by_turn_df_2 = prepare_turn_by_turn_data(base_dir='monitor_responses/turn_by_turn_monitoring_sp5.1')
turn_by_turn_df_3 = prepare_turn_by_turn_data(base_dir='monitor_responses/turn_by_turn_monitoring_sp5.2')
turn_by_turn_df_4 = prepare_turn_by_turn_data(base_dir='monitor_responses/turn_by_turn_monitoring_sp5.3')
combined_results = load_combined_results(include_failed_attempts=False, include_failed_main_task=True)
training_data_path = resolve_path('finetuning/training_data_dfs/combined_3.csv')
training_data = pd.read_csv(training_data_path)
gc.collect()

In [ ]:
EXCLUDED_TASKS = list(training_data.task_name.unique())
def plot_roc_curves_with_excluded_tasks(turn_by_turn_df, combined_results, give_hint=None, include_failed_attempts=False, include_failed_main_task=True, metrics=['final_suspiciousness_turn_by_turn'] + [f'max_suspiciousness_{i}_turn_by_turn' for i in DOWNSAMPLE_FACTORS] + [f'{i}th_highest_suspiciousness_turn_by_turn' for i in [2, 3, 5]] + [f'frac_above_{i}_turn_by_turn' for i in [14, 29, 39]] + [f'{agg_type}_suspiciousness_back_{i}_turn_by_turn' for i in BACK_FRACS for agg_type in ['max', 'mean']], excluded_tasks=EXCLUDED_TASKS):
    plot_roc_curves(turn_by_turn_df[~turn_by_turn_df.task_name.isin(excluded_tasks)], combined_results, give_hint=None, include_failed_attempts=False, include_failed_main_task=True, monitors=turn_by_turn_df.monitor_agent.unique(), metrics=metrics)

In [ ]:
plot_roc_curves_with_excluded_tasks(turn_by_turn_df_1, combined_results)

In [ ]:
plot_roc_curves_with_excluded_tasks(turn_by_turn_df_2, combined_results)

In [ ]:
plot_roc_curves_with_excluded_tasks(turn_by_turn_df_3, combined_results)

In [ ]:
plot_roc_curves_with_excluded_tasks(turn_by_turn_df_4, combined_results)

In [ ]:
for i, df in enumerate([turn_by_turn_df_1, turn_by_turn_df_2, turn_by_turn_df_3, turn_by_turn_df_4]):
    print(f'{i}: {df.max_suspiciousness_1_turn_by_turn.mean():.2f}')

In [ ]:
plot_suspiciousness_distributions(turn_by_turn_df_4, monitors=turn_by_turn_df_4.monitor_agent.unique(), metrics=['final_suspiciousness_turn_by_turn', 'max_suspiciousness_1_turn_by_turn'])